# Этап 2: качество данных

Вход: `data/raw/unified_dataset.csv`. Детект → два варианта `fix` → сравнение.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.data_quality_agent import DataQualityAgent

df = pd.read_csv(ROOT / "data/raw/unified_dataset.csv", encoding="utf-8")
agent = DataQualityAgent(label_column="label", text_column="text")
report = agent.detect_issues(df)
print("n_rows:", report["n_rows"])
print("missing total:", report["missing"]["total_missing_cells"])
print("duplicates (rows in duplicate groups):", report["duplicates"]["n_duplicate_rows"])
print("outliers text len IQR:", report["outliers"]["text_length_iqr"])
print("imbalance:", report["imbalance"].get("counts"))

In [ ]:
# Визуализации: пропуски, классы, длина текста
miss = {k: v["ratio"] for k, v in report["missing"]["by_column"].items()}
plt.figure(figsize=(8, 3))
plt.bar(miss.keys(), miss.values())
plt.xticks(rotation=45, ha="right")
plt.ylabel("доля пропусков")
plt.title("Пропуски по колонкам")
plt.tight_layout()
plt.show()

vc = df["label"].astype(str).value_counts()
plt.figure(figsize=(4, 3))
vc.plot(kind="bar")
plt.title("Распределение label")
plt.tight_layout()
plt.show()

tl = df["text"].astype(str).str.len()
plt.figure(figsize=(6, 3))
plt.hist(tl, bins=40, edgecolor="black", alpha=0.7)
plt.xlabel("длина текста (символы)")
plt.title("Гистограмма длины текста")
plt.tight_layout()
plt.show()

In [ ]:
# Вариант A: balanced (как strategy.yaml по умолчанию)
str_balanced = {
    "missing": "fill_modal_empty",
    "duplicates": "drop",
    "outliers": "clip_iqr_text_length",
    "imbalance": "none",
}
clean_a = agent.fix(df, str_balanced)
cmp_a = agent.compare(df, clean_a)

# Вариант B: conservative — не трогаем длины
str_conservative = {
    "missing": "fill_modal_empty",
    "duplicates": "drop",
    "outliers": "none",
    "imbalance": "none",
}
clean_b = agent.fix(df, str_conservative)
cmp_b = agent.compare(df, clean_b)

cmp_a["variant"] = "balanced"
cmp_b["variant"] = "conservative"
pd.concat([cmp_a, cmp_b], ignore_index=True)

## Обоснование

- **balanced**: убираем дубликаты по `(text, label)`, заполняем пустые `audio`/`image`, отсекаем тексты с экстремальной длиной по IQR — меньше шума для модели, но возможна потеря части длинных новостей.
- **conservative**: сохраняем все длины; если дисбаланс классов критичен, можно добавить `imbalance: undersample_majority` в отдельном прогоне.

Для пайплайна по умолчанию в репозитории зафиксирован **balanced** в `strategy.yaml`.